# Document Q&A Agent

A simple, step-by-step notebook for understanding how the agent validates a document, builds an index, answers a question, and shows source metadata.

Run the cells from top to bottom. The indexing and question cells use OpenAI and require `OPENAI_API_KEY` in `.env`.

## 1. Import the agent functions

In [3]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# The notebook is inside explorations/, so the agent folder is its parent.
agent_folder = Path.cwd().parent
if str(agent_folder) not in sys.path:
    sys.path.insert(0, str(agent_folder))

from agent import (
    answer_question,
    build_index,
    create_chat_engine,
    validate_document_path,
)

load_dotenv(agent_folder / ".env")
print("Agent functions imported.")

Agent functions imported.


## 2. Choose and validate a document

This uses the included PDF sample. Replace the path with any supported file: PDF, DOCX, TXT, Markdown, CSV, JSON, or HTML.

In [4]:
document_path = agent_folder / "documents" / "samples" / "sample_document.pdf"

validated_path = validate_document_path(str(document_path))
print(f"Document: {validated_path.name}")
print(f"Extension: {validated_path.suffix}")
print(f"Exists: {validated_path.is_file()}")

Document: sample_document.pdf
Extension: .pdf
Exists: True


## 3. Build the document index

LlamaIndex loads the document and creates a vector index. This is the first cell that makes an OpenAI embedding request.

In [5]:
index = build_index(str(validated_path))
print("Index is ready.")

📄 Loading and indexing d:\Project\AI_Agent_Agentic_AI_Project\03_Document_QA_Agent\documents\samples\sample_document.pdf...


2026-09-17 21:35:18,022 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


✅ Indexed 1 document chunk(s)
Index is ready.


## 4. Ask one question

The query engine searches the indexed chunks and sends the relevant context to the language model.

In [6]:
question = "What is the workflow?"
answer, sources = answer_question(index, question, validated_path.name)

print(answer)

2026-09-17 21:35:19,434 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-17 21:35:22,754 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The workflow involves validating the document, loading content with LlamaIndex, creating a vector index, and answering with OpenAI.


## 5. Inspect source metadata

Each retrieved source includes the document name, title, page when available, chunk number, and the text used as context.

In [7]:
for source in sources:
    print(f"Chunk: {source['chunk_number']}")
    print(f"Document: {source['document_name']}")
    print(f"Title: {source['title']}")
    print(f"Page: {source['page'] or 'not available'}")
    print(f"Text: {source['text'][:300]}")
    print("-" * 60)

Chunk: 1
Document: sample_document.pdf
Title: sample_document.pdf
Page: 1
Text: Document Q&A Agent Sample
The Document Q&A Agent answers questions about local documents.
Supported formats: PDF, DOCX, TXT, Markdown, CSV, JSON, and HTML.
Workflow: Validate the document, load content with LlamaIndex, create a vector index, and answer with OpenAI.
------------------------------------------------------------


## 6. Optional follow-up chat

The chat engine keeps conversation memory so follow-up questions can refer to earlier context.

In [8]:
chat_engine = create_chat_engine(index)
follow_up = chat_engine.chat("Can you explain that workflow in one sentence?")
print(follow_up.response)

2026-09-17 21:35:30,270 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-17 21:35:32,212 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The workflow involves validating the document, loading its content using LlamaIndex, creating a vector index, and then answering questions using OpenAI.
